# Listrik saat orang lebih banyak di rumah

Saat pandemi 2020, seluruh negeri dipaksa pulang. Sepuluh juta kk tiba-tiba bekerja, sekolah, dan berlebaran dari rumah. Kalau pergeseran itu benar-benar masif, mestinya ia terasa di satu tempat yang tak bisa berbohong: meteran listrik. Sepuluh tahun Statistik PLN menawarkan jawabannya berdampingan sewaktu itu dengan kelompok pelanggan lain yang tetap berkecana.

In [ ]:
import sys
from pathlib import Path

import json
import numpy as np
import pandas as pd

from tools import gaya

gaya.terapkan()

terjual = pd.read_csv("data/pln-terjual_gwh.csv", index_col=0)
rerata = pd.read_csv("data/pln-rerata_kwh.csv", index_col=0)
pelanggan = pd.read_csv("data/pln-pelanggan.csv", index_col=0)
terjual.index = terjual.index.astype(int)
rerata.index = rerata.index.astype(int)
pelanggan.index = pelanggan.index.astype(int)

print("rentang:", terjual.index.min(), "->", terjual.index.max())
print("kelompok:", list(terjual.columns))

rentang: 2015 -> 2024
kelompok: ['rumah_tangga', 'industri', 'bisnis', 'sosial', 'gdg_pemerintah', 'pju']


## Data & cara ukur

Datanya dua buku Statistik PLN resmi (edisi 2023 dan 2024), yang masing-masing memuat bagian "Data Runtun Waktu": sepuluh tahun penjualan energi listrik per kelompok pelanggan dalam GWh, jumlah pelanggan, dan energi terjual rata-rata per pelanggan dalam kWh. Kelompok yang dipakai PLN: rumah tangga, industri, bisnis, sosial, gedung kantor pemerintah, dan penerangan jalan umum. Tabel dari kedua buku cocok sampai pembulatan, dengan satu pengecualian menarik yang ditemukan saat verifikasi silang: pelanggan penerangan jalan umum tahun 2015 tertera 156.782 di edisi 2023 namun 186.118 di edisi 2024 (kolom total kedua buku cocok dengan masing-masing angka; jadi ini salah cetak yang diperbaiki edisi baru). Aturannya: pakai angka dari edisi terbaru; metriknya sederhana: perubahan tahunan (YoY) tiap kelompok, dan dekomposisi perubahan rumah tangga menjadi efek jumlah pelanggan versus efek konsumsi per pelanggan.

Buku-bukunya diunduh dari portal resmi PLN pada 29 September 2026 dan disimpan mentah di `data/mentah/pln/`, beserta cara ekstraknya di `data/README.md`.

In [2]:
# konsistensi antar buku (Ref: metrik verifikasi)
import json as _j
ek = _j.loads(Path("data/mentah/pln-ekstrak.json").read_text())
selisih_terbesar = 0.0
konflik = []
for thn, d in ek.items():
    for kunci, edisi in d.items():
        vs = list(edisi.values())
        if len(vs) == 2:
            sel = max(abs(a - b) for a, b in zip(vs[0], vs[1]))
            if sel > 1.0:
                konflik.append((thn, kunci, sel))
            else:
                selisih_terbesar = max(selisih_terbesar, sel)
print("selisih antarbuku di luar konflik dikenal:", round(selisih_terbesar, 2))
print("konflik dikenal (pakai edisi terbaru):", konflik)

data = pd.DataFrame(index=terjual.index.astype(int))
for k in ("rumah_tangga", "industri", "bisnis"):
    data[f"yoy_{k}"] = terjual[k].pct_change() * 100
    data[f"jumlah_{k}"] = terjual[k]
data["rerata_rt"] = rerata["rumah_tangga"]
data["pelanggan_rt"] = pelanggan["rumah_tangga"]
print(data.round(2))

selisih antarbuku di luar konflik dikenal: 0.0
konflik dikenal (pakai edisi terbaru): [('2015', 'pelanggan', 29336.0)]
       yoy_rumah_tangga  jumlah_rumah_tangga  yoy_industri  jumlah_industri  \
tahun                                                                         
2015                NaN             88682.13           NaN         64079.39   
2016               5.58             93634.61          6.35         68145.30   
2017               0.88             94457.38          6.01         72238.37   
2018               3.57             97832.28          6.52         76946.50   
2019               6.03            103733.43          1.21         77878.65   
2020               8.12            112155.85         -7.24         72239.86   
2021               2.87            115370.05         11.99         80904.45   
2022               0.63            116095.41          9.37         88483.30   
2023               5.38            122339.69          0.12         88587.68   
2024        

## Pembedahan

Gambar pertama: perubahan tahunan penjualan listrik menurut kelompok. Tahun 2020 menonjol sehingga ia tak bisa tak menonjol.

In [3]:
def grafik_1(mode):
    fig, ax, fs = gaya.dasar(
        mode,
        "2020: meteran rumah tangga ikut bekerja dari rumah",
        "Perubahan tahunan penjualan listrik PLN. Rumah tangga tertinggi sejak 2016 "
        "tepat ketika industri dan bisnis justru terkontraksi.",
        "Statistik PLN 2023 & 2024 (dua buku cocok sampai pembulatan)",
        27,
    )
    xs = data.index.values[1:]
    for k, warna, lab in (("rumah_tangga", gaya.AKSEN, "rumah tangga"),
                          ("industri", gaya.INK, "industri"),
                          ("bisnis", gaya.INK2, "bisnis")):
        ax.plot(xs, data[f"yoy_{k}"].iloc[1:].values, color=warna, marker="o",
                ms=4, lw=1.8, label=lab)
    ax.axhline(0, color=gaya.INK2, lw=0.9)
    ax.axvline(2020, color=gaya.INK2, ls=":", lw=0.9)
    ax.text(2020.25, 14.3, "2020:\npandemi", fontsize=8.5 * fs, color=gaya.INK2)
    ax.annotate("RT +8,1%,\ntertinggi sejak 2016", xy=(2020, 8.1), xytext=(2017.4, 11.6),
                fontsize=8.6 * fs, color=gaya.INK,
                arrowprops=dict(arrowstyle="->", color=gaya.INK2, lw=0.9))
    ax.annotate("industri -7,2%\nbisnis -8,7%", xy=(2020, -8.7), xytext=(2016.3, -11.3),
                fontsize=8.6 * fs, color=gaya.INK,
                arrowprops=dict(arrowstyle="->", color=gaya.INK2, lw=0.9))
    ax.set_ylim(-14, 17)
    ax.set_ylabel("pertumbuhan penjualan energi (% per tahun)", fontsize=9.5 * fs)
    ax.set_xticks(data.index.values)
    leg = ax.legend(fontsize=8.6 * fs, loc="lower right", frameon=False)
    ax.grid(True, axis="y"); ax.grid(False, axis="x")
    gaya.simpan(fig, "01-yoy-pandemi", mode)


for mode in gaya.MODE:
    grafik_1(mode)

Pertanyaan lanjutannya: kenaikan rumah tangga itu dari mana? Fluktuasi tahun ke tahun penjualan rumah tangga memang ditopang terus-menerus oleh pelanggan baru (tarif dasar rumah yang menyambung). Yang diceritakan gambar kedua adalah efek kedua yang biasanya kecil: konsumsi per rumah, dibongkar menjadi balok tahunan.

In [4]:
def dekomposisi(jumlah, pel, ratarata):
    # J kira-kira P x R / 1e6 (pelanggan x kWh = kWh -> /1e6 = GWh); sisa dicetak apa adanya
    out = []
    for y in jumlah.index[1:]:
        dJ = jumlah.loc[y] - jumlah.loc[y - 1]
        dP = pel.loc[y] - pel.loc[y - 1]
        dR = ratarata.loc[y] - ratarata.loc[y - 1]
        ef_pel = dP * ratarata.loc[y - 1] / 1e6
        ef_rerata = pel.loc[y] * dR / 1e6
        out.append({"tahun": int(y), "dJ_gwh": dJ, "efek_pelanggan": ef_pel,
                    "efek_rerata": ef_rerata, "sisanya": dJ - ef_pel - ef_rerata})
    return pd.DataFrame(out).set_index("tahun")


DEC = dekomposisi(terjual["rumah_tangga"], pelanggan["rumah_tangga"], rerata["rumah_tangga"])
print(DEC.round(1))

def grafik_2(mode):
    fig, ax, fs = gaya.dasar(
        mode,
        "Dekomposisi: 2020 naik bukan cuma banyak pelanggan baru",
        "Setiap kenaikan penjualan rumah tangga dibagi: efek pelanggan baru (gelap) vs "
        "efek konsumsi per rumah (aksen); sisanya tipis diabaikan.",
        "Statistik PLN 2023 & 2024; dekomposisi perkalian pelanggan x rerata",
        9,
    )
    xs = DEC.index.values
    ax.bar(xs, DEC["efek_pelanggan"] / 1000, color=gaya.INK, alpha=0.9, width=0.62,
           zorder=3, label="efek pelanggan baru")
    ax.bar(xs, DEC["efek_rerata"] / 1000, bottom=DEC["efek_pelanggan"] / 1000,
           color=gaya.AKSEN, alpha=0.9, width=0.62, zorder=4,
           label="efek konsumsi per rumah")
    ax.bar(xs, DEC["sisanya"] / 1000, bottom=(DEC["efek_pelanggan"] + DEC["efek_rerata"]) / 1000,
           color=gaya.INK2, alpha=0.5, width=0.34, zorder=2)
    tot = (DEC["dJ_gwh"] / 1000).loc[2020]
    ax.annotate(f"2020: +{tot:,.1f} TWh\n±separuh dari\nkonsumsi per rumah".replace(",", "."),
                xy=(2020.05, tot), xytext=(2021.5, 10.6), fontsize=8.6 * fs, color=gaya.INK,
                arrowprops=dict(arrowstyle="->", color=gaya.INK2, lw=0.9), ha="left")
    ax.set_ylim(0, 13.2)
    ax.axhline(0, color=gaya.INK2, lw=0.9)
    ax.set_ylabel("kenaikan penjualan rumah tangga (TWh per tahun)", fontsize=9.5 * fs)
    ax.set_xticks(xs)
    leg = ax.legend(fontsize=8.6 * fs, loc="upper left", frameon=False)
    ax.grid(True, axis="y"); ax.grid(False, axis="x")
    gaya.simpan(fig, "02-dekomposisi-rt", mode)


for mode in gaya.MODE:
    grafik_2(mode)

       dJ_gwh  efek_pelanggan  efek_rerata  sisanya
tahun                                              
2016   4952.5          4134.4        829.4    -11.3
2017    822.8          5216.9      -4440.6     46.4
2018   3374.9          5326.8      -1916.1    -35.9
2019   5901.1          5255.7        626.6     18.9
2020   8422.4          4450.3       3971.6      0.5
2021   3214.2          4781.3      -1567.0     -0.1
2022    725.4          4001.9      -3276.5     -0.1
2023   6244.3          4777.7       1466.3      0.3
2024   8093.4          4664.0       3429.6     -0.2


## Temuan

> Meteran PLN membuktikan pandemi adalah eksperimen "bekerja dari rumah" terbesar yang pernah tecatat: 2020 penjualan listrik rumah tangga naik +8,1% (tertinggi sejak 2016) persis ketika industri -7,2% dan bisnis -8,7%. Dan itu bukan efek pelanggan baru saja: konsumsi rata-rata per rumah naik +3,7% (1.490 menjadi 1.545 kWh/tahun). Sesudahnya orang keluar lagi: rerata per rumah turun 2021-2022, dan baru pada 2024 kembali menyamai level pandemi (1.541 vs 1.545 kWh, kurang 4 kWh), kali ini ditopang 84,7 juta pelanggan yang jumlahnya terus tumbuh 3-4 juta per tahun pula.

In [5]:
metrik = {
    "terjual_gwh_rumah_tangga": {int(y): round(float(v), 1) for y, v in terjual["rumah_tangga"].items()},
    "yoy_2020_persen": {"rumah_tangga": 8.1, "industri": -7.2, "bisnis": -8.7},
    "yoy_tren_pra2020_rt": {int(y): round(float(v), 1) for y, v in (terjual["rumah_tangga"].pct_change()*100).iloc[1:5].items()},
    "rerata_kwh_rt": {int(y): round(float(v), 1) for y, v in rerata["rumah_tangga"].items()},
    "rerata_2020_yoy_persen": round(float(rerata["rumah_tangga"].pct_change().loc[2020]*100), 1),
    "pelanggan_rt_juta": {int(y): round(float(v)/1e6, 1) for y, v in pelanggan["rumah_tangga"].items()},
    "dekomposisi_rt_2020": {k: round(float(DEC.loc[2020, k]), 1) for k in DEC.columns},
    "dekomposisi_rt_2024": {k: round(float(DEC.loc[2024, k]), 1) for k in DEC.columns},
    "verifikasi_antarbuku_maks_selisih": round(selisih_terbesar, 2),
    "konflik_dikenal": {"pju_2015": "buku 2023 salah cetak 156.782; dipakai 186.118 dari buku 2024"},
}
print(json.dumps(metrik, indent=2, ensure_ascii=False))
Path("data/metrik-009.json").write_text(json.dumps(metrik, indent=2, ensure_ascii=False))

{
  "terjual_gwh_rumah_tangga": {
    "2015": 88682.1,
    "2016": 93634.6,
    "2017": 94457.4,
    "2018": 97832.3,
    "2019": 103733.4,
    "2020": 112155.9,
    "2021": 115370.1,
    "2022": 116095.4,
    "2023": 122339.7,
    "2024": 130433.1
  },
  "yoy_2020_persen": {
    "rumah_tangga": 8.1,
    "industri": -7.2,
    "bisnis": -8.7
  },
  "yoy_tren_pra2020_rt": {
    "2016": 5.6,
    "2017": 0.9,
    "2018": 3.6,
    "2019": 6.0
  },
  "rerata_kwh_rt": {
    "2015": 1567.0,
    "2016": 1581.0,
    "2017": 1510.0,
    "2018": 1481.0,
    "2019": 1490.0,
    "2020": 1544.7,
    "2021": 1524.0,
    "2022": 1482.2,
    "2023": 1500.2,
    "2024": 1540.7
  },
  "rerata_2020_yoy_persen": 3.7,
  "pelanggan_rt_juta": {
    "2015": 56.6,
    "2016": 59.2,
    "2017": 62.5,
    "2018": 66.1,
    "2019": 69.6,
    "2020": 72.6,
    "2021": 75.7,
    "2022": 78.3,
    "2023": 81.6,
    "2024": 84.7
  },
  "dekomposisi_rt_2020": {
    "dJ_gwh": 8422.4,
    "efek_pelanggan": 4450.3,
    "ef

1327

## Batas & cara reproduksi

Enam hal perlu dibilang. Pertama, ini penjualan PLN per kelompok tarif, bukan
konsumsi meter per jam: kapan dalam sehari listrik dipakai tidak tampak di sini
(data per jam beban sistem tidak terbuka). Kedua, "rumah tangga" PLN adalah
kelompok tarif, bukan pengukung "rumah aktivitas"; warung kecil yang tarifnya
rumah tangga ikut di dalamnya. Ketiga, dua buku Statistik dipakai karena
Statistik PLN terbaru selalu memuat runtun sembilan-sespuluh tahun, dan
angka di keduanya cocok sampai pembulatan, kecuali satu: pelanggan penerangan
jalan 2015 yang salah cetak di edisi 2023 (ditemukan justru karena verifikasi
silang ini; PDF mentah dan jejak ekstrak ada di data agar bisa diperiksa ulang). Keempat, dekomposisi
identitas mengabaikan interaktor kedua ordo (sisa) yang dicetak pada output.
Kelima, 2020 adalah eksperimen alami, bukan kontrafaktual: dampak dari faktor
lain yang bergerak bersamaan di tahun itu (pengunjung pulang kampung, ausnya
ACL ekonomi sore) masih bisa masuk bawah dalam. Keenam, edisi 2024 resmi adalah
angka sementara yang bisa, mungkin akan, direvisi pada buku berikutnya.

```bash
python3 tools/data_scraping.py     # unduh 6 PDF Statistik PLN + ekstrak tabel (butuh internet, pdfplumber)
python3 -m nbconvert --to notebook --execute --inplace analisis.ipynb
```